In [1]:
import subprocess
import tempfile
from itertools import chain
from pathlib import Path

import ghapicache
import githelpers
import release_notes
from IPython.display import Markdown, display
from tqdm.notebook import tqdm

ReleaseMetadata = release_notes.ReleaseMetadata

In [2]:
%load_ext autoreload
%autoreload 1
%aimport githelpers
%aimport ghapicache
%aimport release_notes

In [3]:
local_repo = Path.home() / "Code/celeritas-temp"

# TODO: implement stashed code
# git = githelpers.GitRunner(local_repo)
# release_notes.GIT = git

In [4]:
cached = ghapicache.GhApiCache(repo="celeritas")
ghapi = cached.api
# cached.purge()

In [5]:
ghapi.limit_rem

5000

# Load teams and cache pull request

In [6]:
# Load teams
TEAMS = {}
for t in tqdm(cached.teams()):
    TEAMS[t["name"]] = release_notes.get_team(cached, t)

team_members = frozenset(chain(*(t.members for t in TEAMS.values())))
org_members = frozenset(m["login"] for m in cached.org_members())
TEAMS["affiliates"] = release_notes.Team(
    description="Associated with Celeritas but not core members",
    members=(org_members - team_members),
)

# Initialize the user cache
local_repo = Path("/Users/seth/Code/celeritas-temp")
user_cache = release_notes.UserCache(
    cached, local_repo / "scripts/release/contributors.jsonl"
)
for login in tqdm(org_members):
    user_cache[login]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

In [7]:
user_cache.flush()

In [8]:
!open {user_cache.json_path}

# List active members

This is to be used for crediting in presentations, etc.

In [9]:
text = []
for team, title in [
    ("code-lead", "Code lead"),
    ("core", "Core members"),
    ("core-advisor", "Core advisors"),
    ("affiliates", "Affiliates"),
]:
    text.append(f"## {title}")
    m = [user_cache[username] for username in TEAMS[team].members]
    for member in sorted(m, key=release_notes.get_last_name):
        text.append("- " + release_notes.format_user(member))
    text.append("")

display(Markdown("\n".join(text)))

## Code lead
- Seth R. Johnson *(ORNL)*

## Core members
- Elliott Biondo *(ORNL)*
- Greg Davidson *(ORNL)*
- Julien Esseiva *(LBNL)*
- Rashika Gupta *(ORNL)*
- Hayden Hollenbeck *(UVA)*
- Seth R. Johnson *(ORNL)*
- Soon Yung Jun *(FNAL)*
- Guilherme Lima *(FNAL)*
- Amanda Lund *(ANL)*
- Ben Morgan *(U Warwick)*
- Sakib Rahman *(BNL)*
- Loren Schwiebert *(Wayne State U)*
- Stefano Tognini *(ORNL)*

## Core advisors
- Philippe Canal *(FNAL)*
- Marcel Demarteau *(ORNL)*
- Tom Evans *(ORNL)*

## Affiliates
- None *(None)*
- Lance Bullerwell *(ORNL)*
- Wouter Deconinck *(U Manitoba)*
- Sam Eriksen *(U Bristol)*
- Sanmay Ganguly *(None)*
- Steven Hamilton *(ORNL)*
- Ilker Parmaksiz *(None)*
- Kevin Pedro *(None)*
- Paul Romano *(ANL)*
- Frederic Suter *(ORNL)*
- Sandro Wenzel *(CERN)*


In [12]:
collaborators = [user_cache[username] for username in user_cache.users]

In [ ]:
import re

INST_ADDRESS = {
    "CERN": "CERN, Geneva, Switzerland",
    "ORNL": "Oak Ridge National Laboratory, Oak Ridge, TN, USA",
}


def _normalize_text(value):
    return re.sub(r"\s+", " ", (value or "")).strip()


def ordered_institutes(contributors):
    """Map a contributor list to an ordered, unique set of institutes."""
    preferred = [
        "CERN, Geneva, Switzerland",
        "Oak Ridge National Laboratory, Oak Ridge, TN, USA",
    ]
    ordered = []
    seen = set()

    for preferred_name in preferred:
        preferred_key = _normalize_text(preferred_name).lower()
        for contributor in contributors:
            candidate = _normalize_text(getattr(contributor, "institute", ""))
            if candidate.lower() == preferred_key and candidate.lower() not in seen:
                ordered.append(candidate)
                seen.add(candidate.lower())
                break

    for contributor in contributors:
        candidate = _normalize_text(getattr(contributor, "institute", ""))
        if not candidate:
            continue
        key = candidate.lower()
        if ("cern" in key or "ornl" in key) and key not in seen:
            ordered.append(candidate)
            seen.add(key)

    for contributor in contributors:
        candidate = _normalize_text(getattr(contributor, "institute", ""))
        if not candidate:
            continue
        key = candidate.lower()
        if key not in seen:
            ordered.append(candidate)
            seen.add(key)

    return ordered


def print_chep_authors(contributors, institutes):
    """Print CHEP author blocks given contributors and ordered institute list."""
    institute_map = {inst.lower(): i + 1 for i, inst in enumerate(institutes)}
    author_blocks = []

    for contributor in contributors:
        name = _normalize_text(getattr(contributor, "name", ""))
        if not name:
            continue

        parts = name.split()
        if len(parts) > 1:
            first = " ".join(parts[:-1])
            last = parts[-1]
        else:
            first = ""
            last = parts[0]

        institute = _normalize_text(getattr(contributor, "institute", ""))
        inst_index = institute_map.get(institute.lower())
        inst_suffix = f"\\inst{{{inst_index}}}" if inst_index is not None else ""

        orcid = _normalize_text(getattr(contributor, "orcid", ""))
        orcid_suffix = f"\n  \\orcidlink{{{orcid}}}%" if orcid else ""

        email = _normalize_text(getattr(contributor, "email", ""))
        thanks_suffix = f"\n  \\fnsep\\thanks{{\\email{{{email}}}}}" if email else ""

        author_blocks.append(
            f"  \\firstname{{{first}}} \\lastname{{{last}}}{inst_suffix}%"
            f"{orcid_suffix}{thanks_suffix}"
        )

    if not author_blocks:
        print("  No authors found")
        return

    print("\n  %%\n  \\and\n".join(author_blocks))


def print_chep_institutes(institutes):
    """Print institute addresses joined with \and."""
    addresses = []
    for institute in institutes:
        key = _normalize_text(institute)
        addresses.append(INST_ADDRESS.get(key, key))

    if not addresses:
        print("  No institutes found")
        return

    print("\n  \\and\n".join(f"  {addr}" for addr in addresses))


# Example usage:
institutes = ordered_institutes(collaborators)
print_chep_authors(collaborators, institutes)
print_chep_institutes(institutes)


In [13]:
collaborators[0]._asdict()

{'name': 'Amanda Lund',
 'institute': 'ANL',
 'email': 'alund@anl.gov',
 'orcid': '0000-0002-8316-0709',
 'github': 'amandalund'}

In [14]:
import csv

# Write collaborators to CSV
with open("collaborators.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["last", "first", "orcid", "institution"])

    for collab in sorted(collaborators, key=release_notes.get_last_name):
        # Split name into first and last
        if collab.name:
            name_parts = collab.name.strip().split()
            if len(name_parts) > 1:
                last = name_parts[-1]
                first = " ".join(name_parts[:-1])
            else:
                last = name_parts[0]
                first = ""
        else:
            first = ""
            last = ""

        writer.writerow([last, first, collab.orcid or "", collab.institute or ""])

print(f"Wrote {len(collaborators)} collaborators to collaborators.csv")

Wrote 37 collaborators to collaborators.csv


In [15]:
!open .

In [16]:
# All commits from all time
develop_sha = githelpers.git_rev_parse("upstream/develop")
all_md = ReleaseMetadata(
    release=f"develop-{develop_sha}",
    merge_bases=["v0.0.0"],
    target_branch=develop_sha,
)

all_prs = release_notes.PullRequestRange(all_md)
count_all_contrib = release_notes.ContributionCounter(cached)
for pr in tqdm(all_prs.pull_ids):
    count_all_contrib(pr)
all_contrib_authors = count_all_contrib.sorted().author

Can't match log subject to PR: Format code base (clang-format version 11.0.1)


  0%|          | 0/2040 [00:00<?, ?it/s]

In [17]:
print("Missing ORCIDs:")
print(" ".join(f"@{u}" for u in all_contrib_authors if user_cache[u].orcid is None))

Missing ORCIDs:
@tghaddar @DoaaDeeb


In [18]:
for username in tqdm(set(all_contrib_authors) - set(user_cache.users)):
    print(user_cache[username])

  0%|          | 0/1 [00:00<?, ?it/s]

UserInfo(name=None, institute=None, email=None, orcid=None, github='tghaddar')


In [25]:
cached.flush()
user_cache.flush()

Saved cache to: data/ghapicache-celeritas-project-celeritas.json


# Release process

- Merge base should be all commits *already* released (skip documenting)
- Target branch is the one where the release candidate is
- Previous major branch allows all "v.x" contributors to be credited

In [14]:
# Major release
major_md = release_notes.ReleaseMetadata(
    release="0.7.0", merge_bases=["v0.6.0", "v0.6.3"], target_branch="develop"
)

# Minor release
minor_md = release_notes.ReleaseMetadata(
    release="0.6.3", merge_bases=["v0.6.2"], target_branch="backports/v0.6"
)

## Release note generation

In [15]:
if 0:
    # Backport release:
    release_md = minor_md
    note_body = """
Version {release} is a minor bug fix update.

- Support CUDA 13
- Allow initialization of Geant4 integration before run manager
"""
else:
    # Major release
    release_md = major_md
    note_body = """
Version {release} is a major update to Celeritas featuring:

-

A few minor features are noteworthy:

- 

Notable bug fixes include:

- 

Some interfaces have been removed:

- 

Known deficiencies:

- 
"""

In [16]:
prs = release_notes.PullRequestRange(release_md)
sorted_pulls = release_notes.SortedPulls(cached)
count_contrib = release_notes.ContributionCounter(cached)
for pr_id in tqdm(prs.pull_ids):
    try:
        count_contrib(pr_id)
        sorted_pulls.add(pr_id)
    except Exception as e:
        print(f"Error adding PR #{pr_id}: {e}")
        del cached.cache["pull"][cached.subkey(pr_id)]

reviewers = count_contrib.sorted().reviewer
for login in tqdm(reviewers):
    user_cache[login]

Can't match log subject to PR: Update cmake presets
Can't match log subject to PR: Write setup Geant4 log messages to world logger, not self logger


  0%|          | 0/521 [00:00<?, ?it/s]

Error adding PR #2011: Missing label: #2011 (Add profiling capabilities to `celer-geo`): {'app', 'performance'}
Error adding PR #2017: Missing label: #2017 (Add JLSE env file): {'core'}
Error adding PR #2028: Missing label: #2028 (Finish making JLSE env script POSIX compliant): (no labels)
Error adding PR #2033: Missing label: #2033 (Only set MSVC warning when using MSVC): (no labels)
Error adding PR #2062: Missing label: #2062 (Create .github/.codecov.yml, loosen test requirement): {'core'}
Error adding PR #2078: Missing label: #2078 (Add data() getter functions for the DeviceAllocation class): (no labels)
Error adding PR #2139: Missing label: #2139 (Reduce ORANGE state size for models with background volumes): {'performance', 'orange'}
Error adding PR #2138: Missing label: #2138 (Add ExCL MI300A environment): (no labels)
Error adding PR #2143: Missing label: #2143 (cuda rdc: Check for -z undefs only when CUDA enabled): {'external'}
Error adding PR #2154: Missing label: #2154 (Add LAr

  0%|          | 0/15 [00:00<?, ?it/s]

In [ ]:
def make_notes(cls, release_md, note_body):
    prev = release_md.merge_bases[0]
    other = release_md.merge_bases[1:]
    if not other:
        change_str = f"Changes since {prev} follow."
    else:
        # NOTE: this is redundant if prev is a merge base of other
        assert len(other) == 1
        change_str = (
            f"Changes since {prev}, excluding those released in {other[0]}, follow."
        )

    notes = cls(release_md, note_body)
    notes.paragraph(change_str)
    notes.sorted_pulls(sorted_pulls)
    notes.reviewers(reviewers, user_cache)
    notes.changelog_line("celeritas-project", "celeritas")

    return notes

In [15]:
rst_notes = make_notes(release_notes.RstNotes, release_md, note_body)

with githelpers.open_pbcopy() as pb:
    rst_notes.write(pb)

print("Release notes copied to clipboard")
subprocess.run(["open", str(githelpers.REPO / "doc/release-history")])

Release notes copied to clipboard


CompletedProcess(args=['open', '/Users/seth/Code/celeritas-temp/doc/release-history'], returncode=0)

## Create github release

For backport releases, make sure the release history above is *first* merged into upstream **and cherry-picked into the backport branch**.

First, tag the release locally and update the target to the tagged commit.

In [ ]:
release_notes.create_local_tag(release_md)
subject = githelpers.git("log", "-1", r"--format=%s", release_md.target_branch)[0]
if not subject.startswith("Release v" + release_md.release):
    print(
        f"WARNING: expected last commit subject to include release, but subject line is '{subject}'"
    )

Updating release target from backports/v0.6 to ddf6c690e37d53b56d001e6bf0c26cb1b14963fa


Then create a gzipped in-memory archive of the repository. Decompress and run `cmake -P cmake/CgvFindVersion.cmake` to check the version is exported correctly.

In [17]:
tgz_data = githelpers.git_archive_tgz(release_md.target_branch)
tgz_name = f"{cached.repo}-{release_md.release}.tar.gz"

with tempfile.TemporaryDirectory(delete=False) as tmpdir:
    with open(Path(tmpdir) / tgz_name, "wb") as f:
        f.write(tgz_data)
    subprocess.run(["open", tmpdir])

In [ ]:
subprocess.run(
    ["cmake", "-P", "cmake/CgvFindVersion.cmake"],
    cwd=Path(tmpdir) / f"{cached.repo}-{release_md.release}",
)

Now generate the PDF documentation and check it.

In [ ]:
build_dir = local_repo / "build"


def run_build(*args):
    return subprocess.run(args, cwd=build_dir, check=True)


run_build("cmake", "-UCeleritas_GIT_DESCRIBE", ".")
run_build("ninja", "doc_pdf")
pdf_doc = build_dir / "doc" / "latex" / "Celeritas.pdf"
assert pdf_doc.is_file()
!open {pdf_doc}

-- Configuring done (0.5s)
-- Generating done (0.2s)
-- Build files have been written to: /Users/seth/Code/celeritas-temp/build
[1/6] Performing update step for 'ornltm'
-- Already at requested ref: 6379032d2f1e5e8bc1589d087465371fd8b432c1
[2/6] No patch step for 'ornltm'
[3/6] No configure step for 'ornltm'
[4/6] No build step for 'ornltm'
[5/6] No install step for 'ornltm'
[6/6] Completed 'ornltm'


Create the github release draft and upload the tarball and PDF documentation

In [ ]:
ghapi = cached.api

markdown_notes = make_notes(release_notes.MarkdownNotes, release_md, note_body)

assert release_md.release is not None
gh_release = release_notes.find_release(ghapi, release_md.release)
if not gh_release:
    gh_release = release_notes.create_release(ghapi, release_md, str(markdown_notes))
    tgz_asset = release_notes.upload_asset(cached, gh_release, tgz_data, tgz_name)
    doc_asset = release_notes.upload_asset(
        cached, gh_release, pdf_doc, f"celeritas-user-manual-{release_md.release}.pdf"
    )

No release found for version: 0.6.3
Draft release 0.6.3 created: https://github.com/celeritas-project/celeritas/releases/tag/untagged-804e7f799a631819ea29
Uploaded artifact: https://github.com/celeritas-project/celeritas/releases/download/untagged-804e7f799a631819ea29/celeritas-0.6.3.tar.gz
Uploaded artifact: https://github.com/celeritas-project/celeritas/releases/download/untagged-804e7f799a631819ea29/celeritas-user-manual-0.6.3.pdf


In [ ]:
sha256_hash = release_notes.prepare_spack_update("g4vg", release_md, tgz_data)

Spack version copied to clipboard!


0

In [ ]:
display(
    Markdown(f"""
## Time to publish!

1. Visit [the github release page]({gh_release["html_url"]})
2. Verify the tgz and pdf are both created
3. Ensure the tgz SHA256 sum is {sha256_hash}
4. Double check the name and release notes
5. Edit and publish!
""")
)


## Time to publish

1. Visit [the github release page](https://github.com/celeritas-project/celeritas/releases/tag/untagged-804e7f799a631819ea29)
2. Verify the tgz and pdf are both created
3. Ensure the tgz SHA256 sum is c87910acd0bb9dc85c53e2e4d9958f36d3bb1e5975f0e08fe7274b332a2ccf27
4. Double check the name and release notes
5. Hit publish!


## Print authors since the last major release for presentations

In [ ]:
latest_major_md = ReleaseMetadata(
    release=f"recent-{release_md.target_branch}",
    merge_bases=["v0.7.0-dev"],
    target_branch="upstream/develop",
    # merge_bases=['v0.6.0-dev'],
    # target_branch='backports/v0.6',
)

latest_prs = release_notes.PullRequestRange(latest_major_md)
count_contrib = release_notes.ContributionCounter(cached)
for pr in tqdm(latest_prs.pull_ids):
    count_contrib(pr)
authors = count_contrib.sorted().author

text = ["# Current contributors"]
current_contrib = set()
for username in authors:
    m = user_cache[username]
    text.append(f"- {m.name} (@{m.github})")
    current_contrib.add(m.github)
text.extend(["", "# Past contributors"])
for username in all_contrib_authors:
    if username not in current_contrib:
        m = user_cache[username]
        text.append(f"- {m.name} (@{username})")
# display(Markdown("\n".join(text)))
print("\n".join(text))

  0%|          | 0/494 [00:00<?, ?it/s]

# Current contributors
- Seth R. Johnson (@sethrj)
- Amanda Lund (@amandalund)
- Elliott Biondo (@elliottbiondo)
- Julien Esseiva (@esseivaju)
- Hayden Hollenbeck (@hhollenb)
- Rashika Gupta (@Rashika-Gupta)
- Philippe Canal (@pcanal)
- Stefano Tognini (@stognini)
- Loren Schwiebert (@LSchwiebert)
- Soon Yung Jun (@whokion)
- Greg Davidson (@davidsgr)
- Owen Strong (@osanstrong)
- Ben Morgan (@drbenmorgan)
- Sakib Rahman (@rahmans1)
- Guilherme Lima (@mrguilima)
- Lance Bullerwell (@lebuller)
- Stephen Nicholas Swatman (@stephenswat)

# Past contributors
- Paul Romano (@paulromano)
- Vincent R. Pascuzzi (@vrpascuzzi)
- Tom Evans (@tmdelellis)
- Vidor Heli Lujan Montiel (@VHLM2001)
- Damien L-G (@dalg24)
- Doaa Deeb (@DoaaDeeb)
- Andrey Prokopenko (@aprokop)
- Shane Hart (@hartsw)
- Peter Heywood (@ptheywood)


In [19]:
display(Markdown("\n".join(text)))

# Current contributors
- Seth R. Johnson (@sethrj)
- Amanda Lund (@amandalund)
- Elliott Biondo (@elliottbiondo)
- Julien Esseiva (@esseivaju)
- Hayden Hollenbeck (@hhollenb)
- Rashika Gupta (@Rashika-Gupta)
- Philippe Canal (@pcanal)
- Stefano Tognini (@stognini)
- Loren Schwiebert (@LSchwiebert)
- Soon Yung Jun (@whokion)
- Greg Davidson (@davidsgr)
- Owen Strong (@osanstrong)
- Ben Morgan (@drbenmorgan)
- Sakib Rahman (@rahmans1)
- Guilherme Lima (@mrguilima)
- Lance Bullerwell (@lebuller)
- Stephen Nicholas Swatman (@stephenswat)

# Past contributors
- Paul Romano (@paulromano)
- Vincent R. Pascuzzi (@vrpascuzzi)
- Tom Evans (@tmdelellis)
- Vidor Heli Lujan Montiel (@VHLM2001)
- Damien L-G (@dalg24)
- Doaa Deeb (@DoaaDeeb)
- Andrey Prokopenko (@aprokop)
- Shane Hart (@hartsw)
- Peter Heywood (@ptheywood)

In [18]:
cached.flush()

Saved cache to: data/ghapicache-celeritas-project-celeritas.json


## Push to zenodo

In [ ]:
import zenodoapi

%aimport zenodoapi

In [26]:
# Load the Zenodo token
token_path = Path.home() / ".config/zenodo-token"
with open(token_path) as f:
    zenodo_token = f.read().strip()

zenodo = zenodoapi.Zenodo(zenodo_token)
zenodo.api_url = "https://zenodo.org/api/"

In [ ]:
def load_contributions(release_md):
    prs = release_notes.PullRequestRange(release_md)
    count_contrib = release_notes.ContributionCounter(cached)
    for pr in tqdm(prs.pull_ids):
        count_contrib(pr)

    # Create author list
    return count_contrib.sorted()


def download_tarball(gh_release):
    asset = gh_release["assets"][0]
    name = asset["name"]
    content = cached.download_file(asset["url"], ext=Path(name).suffix)
    assert content is not None
    return (content, name)


make_zenodo_md = release_notes.ZenodoMetadataBuilder(user_cache=user_cache, teams=TEAMS)

### Minor release

This assumes the major release has already been published upstream into Zenodo.

In [28]:
assert not release_md.is_major()
major, minor, patch = release_md.as_version()
release_md = release_notes.ReleaseMetadata.from_comprehensive_version(minor, patch)
major_dep = zenodo.find_deposition(f"Celeritas {major}.{minor}")

In [29]:
zenodo.get_deposition(major_dep.get_latest_version().id)

ZenodoDeposition(id=17193315, title="Celeritas 0.6")

In [30]:
gh_release = release_notes.find_release(ghapi, release_md.release)

In [ ]:
# Get the Zenodo metadata
from requests import HTTPError


old_md = major_dep.data["metadata"]
new_md = make_zenodo_md(load_contributions(release_md), release_md, gh_release)
# Only update editors, not team members
old_contrib = new_md["contributors"]
new_contrib = [u for u in old_contrib if u["type"] == "Editor"]
old_contrib = [u for u in old_contrib if u["type"] != "Editor"]
new_md["contributors"] = new_contrib + old_contrib
# Don't change the title
new_md["title"] = major_dep.data["metadata"]["title"]
# Add the body
new_md["description"] = "\n\n".join([
    old_md["description"],
    f"<h1>Version {release_md.release}</h1>",
    new_md["description"],
])
try:
    new_vers = major_dep.create_new_version()
except HTTPError as e:
    # Maybe the version already exists
    if e.response.status_code == 400:
        print("Draft may already exist:", e)
        new_vers = major_dep.get_latest_draft()
    else:
        raise
try:
    new_vers.update(new_md)
except HTTPError as e:
    # Something else went wrong? Try to upload the tarball
    print("Draft may already exist:", e)

# Upload the release
(content, name) = download_tarball(gh_release)
new_vers.upload(content, name)
# The old tarball may still be there (this is buggy) so delete it
new_vers.refresh()
for file in new_vers.get_files():
    if file.filename != name:
        try:
            file.delete()
        except Exception as e:
            print(f"Failed to delete {file.filename}: {e}")
new_vers.refresh()

Can't match log subject to PR: Update cmake presets
Can't match log subject to PR: Write setup Geant4 log messages to world logger, not self logger


  0%|          | 0/369 [00:00<?, ?it/s]

Created new version 17684516: https://zenodo.org/deposit/17684516
Updated deposition at https://zenodo.org/deposit/17684516 : Celeritas 0.6
Uploaded celeritas-0.6.3.tar.gz: version 3056d2d2-889c-4044-8db5-631e0f197235
Failed to delete celeritas-0.6.2.tar.gz: Expecting value: line 1 column 1 (char 0)


ZenodoDeposition(id=17684516, title="Celeritas 0.6")

## Major release

In [ ]:
assert release_md.is_major()
major, minor, patch = release_md.as_version()
release_md = release_notes.ReleaseMetadata.from_comprehensive_version(minor)
title = f"Celeritas {major}.{minor}"

In [ ]:
zmd = make_zenodo_md(load_contributions(release_md), release_md, gh_release)
zmd["title"] = title

# Create the Zenodo deposition
deposition = zenodo.create_deposition(zmd)

  0%|          | 0/234 [00:00<?, ?it/s]

Created deposition 15281110 at https://zenodo.org/deposit/15281110 : Celeritas 0.6


In [ ]:
# Upload the release tarball
deposition.upload(artifact_tgz, f"celeritas-{release_md.release}.tar.gz")

print(deposition.html)
print("""
IMPORTANT Checklist:
- Add community (CANNOT be done later)
- Update contributors based on release date
- Update licenses
""")

Uploaded celeritas-0.6.0.tar.gz: version bbd63016-30a1-4be9-9a74-329ca975bed9
https://zenodo.org/deposit/15281110

IMPORTANT Checklist:
- Add community (CANNOT be done later)
- Update contributors based on release date
- Update licenses

